🌍 TravelGenie AI

Intelligent Multi-Agent Travel Planner

Install Libraries

In [1]:
!pip install -q openai gradio requests pandas

Import Libraries

In [2]:
import requests
import pandas as pd
import gradio as gr

from openai import OpenAI
from google.colab import userdata

Load Groq API

In [3]:
GROQ_API_KEY = userdata.get("GROQ_API_KEY")

client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
)

Memory

In [4]:
memory = {}

Planner Agent

This agent understands the user's travel request and creates an execution plan.

In [5]:
def planner_agent(task):

    prompt = f"""
You are an Expert AI Travel Planner.

Your responsibilities:

1. Understand the user's travel request.
2. Break the task into logical steps.
3. Identify what information is needed.
4. Return ONLY a numbered execution plan.

User Request:

{task}
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": "You are an Expert AI Travel Planner."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2,
        max_tokens=1000
    )

    plan = response.choices[0].message.content.strip()

    memory["task"] = task
    memory["plan"] = plan

    return plan

Test Planner Agent

In [6]:
task = """
Plan a 5-day family trip.

Destination: Ooty

Starting City: Chennai

Budget: ₹30,000

Travel Month: December
"""

print("=" * 60)
print("PLANNER AGENT")
print("=" * 60)

plan = planner_agent(task)

print(plan)

PLANNER AGENT
1. Determine the number of family members traveling to calculate accommodation and transportation costs.
2. Research and book a suitable mode of transportation from Chennai to Ooty (train, bus, or car) within the given budget.
3. Explore Ooty's top attractions and activities suitable for a 5-day family trip in December, considering weather conditions.
4. Shortlist budget-friendly accommodations in Ooty (hotels, resorts, or homestays) that fit the family's size and budget.
5. Create a daily itinerary for the 5-day trip, including travel time, sightseeing, and leisure activities.
6. Estimate food and miscellaneous expenses for the trip to ensure the total cost stays within ₹30,000.
7. Book accommodations and transportation in advance to avoid peak season rates and availability issues.
8. Research any additional costs, such as entry fees for attractions or activities, and factor them into the overall budget.


Memory Check

After running the test, verify that the planner stored its output:



In [7]:
print(memory)

{'task': '\nPlan a 5-day family trip.\n\nDestination: Ooty\n\nStarting City: Chennai\n\nBudget: ₹30,000\n\nTravel Month: December\n', 'plan': "1. Determine the number of family members traveling to calculate accommodation and transportation costs.\n2. Research and book a suitable mode of transportation from Chennai to Ooty (train, bus, or car) within the given budget.\n3. Explore Ooty's top attractions and activities suitable for a 5-day family trip in December, considering weather conditions.\n4. Shortlist budget-friendly accommodations in Ooty (hotels, resorts, or homestays) that fit the family's size and budget.\n5. Create a daily itinerary for the 5-day trip, including travel time, sightseeing, and leisure activities.\n6. Estimate food and miscellaneous expenses for the trip to ensure the total cost stays within ₹30,000.\n7. Book accommodations and transportation in advance to avoid peak season rates and availability issues.\n8. Research any additional costs, such as entry fees for

Research Agent

This agent will convert the plan into useful travel information

Destination Research Agent, which will gather information about:

Best tourist attractions
Climate
Local food
Transportation
Shopping
Best time to visit

In [8]:
def research_agent(task, plan):

    prompt = f"""
You are an Expert Travel Research Agent.

Your responsibilities:

1. Read the travel request.
2. Read the execution plan.
3. Research the destination.

Provide the following:

1. Destination Overview
2. Best Time to Visit
3. Famous Tourist Attractions
4. Local Foods to Try
5. Transportation Options
6. Shopping Places
7. Travel Tips

Return the information in a well-formatted report.

Travel Request:
{task}

Execution Plan:
{plan}
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": "You are an Expert Travel Research Agent."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.3,
        max_tokens=2000
    )

    research = response.choices[0].message.content.strip()

    memory["research"] = research

    return research

Test Research Agent

In [9]:
print("="*60)
print("RESEARCH AGENT")
print("="*60)

research = research_agent(task, plan)

print(research)

RESEARCH AGENT
**Ooty Family Trip Report**

### Destination Overview

Ooty, also known as Udhagamandalam, is a popular hill station in the Nilgiri Hills of Tamil Nadu, India. It is known for its stunning natural beauty, pleasant climate, and rich cultural heritage. Ooty is an ideal destination for a family trip, offering a range of activities and attractions that cater to all ages.

### Best Time to Visit

The best time to visit Ooty is from October to February, with December being a great month to experience the town's festive atmosphere and mild winter climate. The average temperature in December ranges from 8°C to 15°C, making it perfect for outdoor activities and sightseeing.

### Famous Tourist Attractions

1. **Ooty Lake**: A scenic lake with boating facilities and a popular spot for picnics.
2. **Doddabetta Peak**: The highest point in the Nilgiri Hills, offering breathtaking views of the surrounding landscape.
3. **Botanical Gardens**: A beautiful garden with a wide variety of 

Check Memory

In [10]:
print(memory.keys())

dict_keys(['task', 'plan', 'research'])


Weather Agent

In [13]:
from google.colab import userdata

key = userdata.get("WEATHER_API_KEY")

print(key[:8])     # Prints only the first 8 characters
print(len(key))

fd00aa32
32


In [14]:
from google.colab import userdata
import requests

WEATHER_API_KEY = userdata.get("WEATHER_API_KEY")

url = "https://api.openweathermap.org/data/2.5/weather"

params = {
    "q": "Ooty",
    "appid": WEATHER_API_KEY,
    "units": "metric"
}

response = requests.get(url, params=params)

print(response.status_code)
print(response.json())

401
{'cod': 401, 'message': 'Invalid API key. Please see https://openweathermap.org/faq#error401 for more info.'}


Budget Agent

In [15]:
def budget_agent(task, research):

    prompt = f"""
You are an AI Budget Planner.

Responsibilities:

1. Estimate transportation cost.
2. Estimate hotel cost.
3. Estimate food cost.
4. Estimate sightseeing cost.
5. Estimate shopping cost.
6. Give the total estimated budget.

Return a neat report.

Travel Request:

{task}

Research:

{research}
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role":"system",
                "content":"You are an Expert Travel Budget Planner."
            },
            {
                "role":"user",
                "content":prompt
            }
        ],
        temperature=0.2,
        max_tokens=1200
    )

    budget = response.choices[0].message.content.strip()

    memory["budget"] = budget

    return budget

In [16]:
print("="*60)
print("BUDGET AGENT")
print("="*60)

budget = budget_agent(task, research)

print(budget)

BUDGET AGENT
**Ooty Family Trip Budget Report**

### Introduction

This report provides a detailed breakdown of the estimated costs for a 5-day family trip to Ooty, starting from Chennai, within a budget of ₹30,000.

### Estimated Costs

1. **Transportation Cost**: ₹8,000
	* This includes the cost of train or bus fare from Chennai to Ooty, as well as any additional transportation costs within Ooty.
2. **Hotel Cost**: ₹10,000
	* This includes the cost of a budget-friendly hotel or homestay for 5 nights, with an average cost of ₹2,000 per night.
3. **Food Cost**: ₹4,000
	* This includes the estimated cost of meals, snacks, and beverages for 5 days, with an average cost of ₹800 per day.
4. **Sightseeing Cost**: ₹2,000
	* This includes the estimated cost of entry fees, boating, and other activities at tourist attractions.
5. **Shopping Cost**: ₹2,000
	* This includes the estimated cost of souvenirs, handicrafts, and other shopping expenses.

### Total Estimated Budget

The total estimated 

Memory

In [17]:
print(memory.keys())

dict_keys(['task', 'plan', 'research', 'budget'])


🗓️ Itinerary Agent

This agent generates a complete day-wise travel plan.

In [18]:
def itinerary_agent(task, research, budget):

    prompt = f"""
You are an Expert Travel Itinerary Planner.

Your responsibilities:

1. Read the travel request.
2. Read the destination research.
3. Read the estimated budget.
4. Create a detailed day-wise itinerary.

Include:

• Morning
• Afternoon
• Evening
• Recommended food
• Estimated daily expense
• Travel tips

Travel Request:
{task}

Research:
{research}

Budget:
{budget}

Return a professional itinerary.
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role":"system",
                "content":"You are an Expert Travel Planner."
            },
            {
                "role":"user",
                "content":prompt
            }
        ],
        temperature=0.3,
        max_tokens=2500
    )

    itinerary = response.choices[0].message.content.strip()

    memory["itinerary"] = itinerary

    return itinerary

In [19]:
print("="*60)
print("ITINERARY AGENT")
print("="*60)

itinerary = itinerary_agent(
    task,
    research,
    budget
)

print(itinerary)

ITINERARY AGENT
**Ooty Family Trip Itinerary**

### Day 1: Chennai to Ooty

* **Morning**: Depart from Chennai by train (Nilgiri Express) or bus to Mettupalayam. From Mettupalayam, take a toy train to Ooty.
* **Afternoon**: Check-in to the hotel and freshen up. Visit the **Ooty Lake** for boating and a picnic.
* **Evening**: Explore the **Charring Cross** shopping area and try some **Nilgiri Tea** at a local tea stall.
* **Recommended Food**: Try some **South Indian Cuisine** at a local restaurant for dinner.
* **Estimated Daily Expense**: ₹4,500 (transportation: ₹2,000, hotel: ₹1,500, food: ₹800, sightseeing: ₹200)
* **Travel Tips**: Book your train or bus tickets in advance to avoid peak season rates. Pack warm clothing for the winter months.

### Day 2: Ooty

* **Morning**: Visit the **Doddabetta Peak** for breathtaking views of the surrounding landscape.
* **Afternoon**: Explore the **Botanical Gardens** and enjoy the tranquil atmosphere.
* **Evening**: Visit the **Wenlock Downs** 

Check Memory

In [20]:
print(memory.keys())

dict_keys(['task', 'plan', 'research', 'budget', 'itinerary'])


Reviewer Agent

This agent reviews the complete travel plan before presenting it to the user.

In [21]:
def reviewer_agent(plan, research, budget, itinerary):

    prompt = f"""
You are a Senior Travel Planner.

Review the complete travel plan.

Check:

1. Is the itinerary realistic?
2. Is the budget reasonable?
3. Are important tourist places included?
4. Are travel tips useful?
5. Are there missing recommendations?
6. Suggest improvements if needed.

Return:

Overall Rating (1-10)

Strengths

Weaknesses

Suggestions

Travel Plan

{plan}

Research

{research}

Budget

{budget}

Itinerary

{itinerary}
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role":"system",
                "content":"You are an Expert Travel Reviewer."
            },
            {
                "role":"user",
                "content":prompt
            }
        ],
        temperature=0.2,
        max_tokens=1800
    )

    review = response.choices[0].message.content.strip()

    memory["review"] = review

    return review

Test Reviewer Agent

In [22]:
print("="*60)
print("REVIEWER AGENT")
print("="*60)

review = reviewer_agent(
    plan,
    research,
    budget,
    itinerary
)

print(review)

REVIEWER AGENT
**Overall Rating: 8/10**

**Strengths:**

1. The travel plan is well-structured and easy to follow.
2. The itinerary includes a good mix of sightseeing, adventure, and relaxation activities.
3. The budget breakdown is detailed and realistic.
4. The travel tips and recommendations are helpful and practical.

**Weaknesses:**

1. The itinerary is a bit packed, with some days having multiple activities that may be tiring for a family trip.
2. The budget breakdown does not include any contingency funds for unexpected expenses.
3. Some activities, such as the cultural show and campfire, may not be suitable for all family members.
4. The itinerary does not include any free time for relaxation or spontaneity.

**Suggestions:**

1. Consider adding some free time to the itinerary to allow for relaxation or spontaneity.
2. Include contingency funds in the budget breakdown to account for unexpected expenses.
3. Provide more options for activities that cater to different interests an

In [23]:
print(memory.keys())

dict_keys(['task', 'plan', 'research', 'budget', 'itinerary', 'review'])


build the Main AI Controller

Instead of calling every agent manually, we'll create one function:

Master Agent (Orchestrator) that connects everything together.

In [24]:
def travel_genie(destination, days, budget, travel_type):

    task = f"""
Destination : {destination}

Number of Days : {days}

Budget : ₹{budget}

Travel Type : {travel_type}
"""

    # Planner
    plan = planner_agent(task)

    # Research
    research = research_agent(task, plan)

    # Budget
    budget_report = budget_agent(task, research)

    # Itinerary
    itinerary = itinerary_agent(
        task,
        research,
        budget_report
    )

    # Reviewer
    review = reviewer_agent(
        plan,
        research,
        budget_report,
        itinerary
    )

    final_report = f"""
# 🌍 TravelGenie AI

========================================

## 📋 Execution Plan

{plan}

========================================

## 🔍 Destination Research

{research}

========================================

## 💰 Budget Estimate

{budget_report}

========================================

## 🗓 Day-wise Itinerary

{itinerary}

========================================

## ✅ Reviewer Feedback

{review}

========================================
"""

    return final_report

Test the Entire AI

In [25]:
result = travel_genie(
    destination="Ooty",
    days=5,
    budget=30000,
    travel_type="Family"
)

print(result)


# 🌍 TravelGenie AI


## 📋 Execution Plan

1. Determine the best time to visit Ooty based on weather and tourist season to plan accordingly.
2. Research and book suitable family-friendly accommodations in Ooty within the given budget of ₹30000 for 5 days.
3. Identify top family-friendly attractions and activities in Ooty, such as Ooty Lake, Botanical Gardens, and Doddabetta Peak.
4. Plan a daily itinerary for the 5-day trip, including travel time between attractions and potential downtime for relaxation.
5. Calculate the estimated cost of food, transportation, and entry fees for each attraction to ensure the trip stays within budget.
6. Research and book transportation to and from Ooty, including flights, trains, or buses, and local transportation options such as taxis or car rentals.
7. Consider booking a guided tour or hiring a local guide to help navigate Ooty and provide insight into the local culture and history.
8. Purchase travel insurance to cover unexpected medical or travel-r

Planner
      ↓
Research
      ↓
Budget
      ↓
Itinerary
      ↓
Reviewer

Build the Gradio Interface

In [ ]:
import gradio as gr

demo = gr.Interface(
    fn=travel_genie,

    inputs=[
        gr.Textbox(
            label="📍 Destination",
            placeholder="e.g. Ooty"
        ),

        gr.Number(
            label="📅 Number of Days",
            value=5
        ),

        gr.Number(
            label="💰 Budget (₹)",
            value=30000
        ),

        gr.Dropdown(
            ["Solo", "Family", "Friends", "Couple"],
            value="Family",
            label="👨‍👩‍👧 Travel Type"
        )
    ],

    outputs=gr.Markdown(label="🌍 Travel Plan"),

    title="🌍 TravelGenie AI",

    description="""
### Intelligent Multi-Agent Travel Planner

Powered by:

• Planner Agent, • Research Agent, • Budget Agent, • Itinerary Agent, • Reviewer Agent

"""
)

demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://fe6caaba40c38dd7ec.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


🌍 TravelGenie AI
========================================

📋 Execution Plan
Gather information about Kodaikanal, including its location, climate, and tourist attractions.
Determine the best time to visit Kodaikanal based on the family's preferences and budget.
Identify suitable accommodation options in Kodaikanal that fit within the ₹15000 budget for 3 days.
Research and shortlist family-friendly activities and attractions in Kodaikanal, such as trekking, boating, and sightseeing.
Create a daily itinerary for the 3-day trip, including travel arrangements, meal plans, and activity schedules.
Calculate the estimated costs for transportation, food, and activities to ensure they are within the ₹15000 budget.
Provide recommendations for transportation options from the user's location to Kodaikanal, such as flights, trains, or buses.
Book accommodations and make reservations for activities and attractions in advance to avoid peak season crowds.
Prepare a list of essential items to pack for the trip, including clothing, toiletries, and medications.
Finalize the travel plan and provide a detailed itinerary to the user.
========================================

🔍 Destination Research
Kodaikanal Travel Report
Destination Overview
Kodaikanal is a picturesque hill station located in the Dindigul district of Tamil Nadu, India. It is situated at an altitude of 2,133 meters above sea level and is known for its stunning natural beauty, with lush green forests, sparkling lakes, and rolling hills. The town has a rich history and was established by the British in 1845 as a summer resort.

Best Time to Visit
The best time to visit Kodaikanal is from September to May, when the weather is pleasant and cool. The summer months (June to August) are ideal for trekking and outdoor activities, while the winter months (December to February) are perfect for relaxing and enjoying the scenic beauty of the town.

Famous Tourist Attractions
Kodaikanal Lake: A man-made lake that offers boating and fishing facilities.
Coaker's Walk: A scenic walking path that offers breathtaking views of the surrounding hills and valleys.
Pillar Rocks: A unique rock formation that is a popular spot for trekking and photography.
Bryant Park: A beautiful park that is home to a variety of flora and fauna.
Silver Cascade: A stunning waterfall that is a popular spot for picnics and relaxation.
Local Foods to Try
South Indian cuisine: Kodaikanal is known for its delicious South Indian dishes, including idlis, dosas, and vadas.
Tamil Nadu specialties: Try the local specialties, such as parottas, chettinad chicken, and fish fry.
Fresh fruits and vegetables: Kodaikanal is known for its fresh produce, including strawberries, carrots, and beans.
Homemade chocolates: The town is famous for its homemade chocolates, which make for a delicious souvenir.
Transportation Options
Flights: The nearest airport is the Madurai Airport, which is located about 120 km from Kodaikanal.
Trains: The nearest railway station is the Kodaikanal Road station, which is located about 80 km from Kodaikanal.
Buses: Regular bus services are available from major cities, including Chennai, Bangalore, and Madurai.
Taxis and autorickshaws: Taxis and autorickshaws are available for local transportation.
Shopping Places
Kodaikanal Market: A bustling market that sells everything from fresh produce to handicrafts.
PT Road: A popular shopping street that is lined with shops, restaurants, and cafes.
Handicraft shops: Kodaikanal is known for its handicrafts, including wooden carvings, pottery, and textiles.
Travel Tips
Book accommodations in advance: Kodaikanal is a popular tourist destination, and accommodations can fill up quickly.
Pack warm clothing: The weather in Kodaikanal can be cool, especially in the winter months.
Bring sunscreen and sunglasses: The sun can be strong in Kodaikanal, especially during the summer months.
Respect the local environment: Kodaikanal is a fragile ecosystem, and visitors are encouraged to respect the local environment and wildlife.
Budget Breakdown

Accommodation: ₹6,000 (avg. ₹2,000 per night for 3 nights)
Food: ₹3,000 (avg. ₹1,000 per day for 3 days)
Transportation: ₹2,000 (depending on the mode of transport)
Activities: ₹2,000 (avg. ₹667 per day for 3 days)
Total: ₹13,000
This leaves a buffer of ₹2,000 for any unexpected expenses or additional activities.

========================================

💰 Budget Estimate
Kodaikanal Travel Budget Report
Destination: Kodaikanal
Number of Days: 3
Travel Type: Family
Budget: ₹15,000
Estimated Costs:
Transportation Cost: ₹2,000 * This estimate is based on the cost of transportation from the nearest airport or railway station to Kodaikanal, as well as local transportation costs.
Hotel Cost: ₹6,000 * This estimate is based on an average cost of ₹2,000 per night for a family-friendly hotel in Kodaikanal.
Food Cost: ₹3,000 * This estimate is based on an average cost of ₹1,000 per day for meals and snacks for a family.
Sightseeing Cost: ₹2,000 * This estimate is based on the cost of visiting popular tourist attractions in Kodaikanal, such as Kodaikanal Lake, Coaker's Walk, and Pillar Rocks.
Shopping Cost: ₹1,000 * This estimate is based on the cost of shopping for souvenirs and local handicrafts in Kodaikanal.
Total Estimated Budget: ₹14,000
This leaves a buffer of ₹1,000 for any unexpected expenses or additional activities.

Recommendations:
Book accommodations and transportation in advance to avoid high costs.
Try local cuisine and street food to save on food costs.
Plan sightseeing activities carefully to avoid unnecessary expenses.
Shop for souvenirs and handicrafts at local markets to get the best deals.
By following these recommendations, you can have a enjoyable and budget-friendly trip to Kodaikanal with your family.

========================================

🗓 Day-wise Itinerary
Kodaikanal Family Trip Itinerary

Day 1: Arrival and Exploration

Morning: Arrive at Madurai Airport and take a taxi or bus to Kodaikanal (approximately 3-4 hours). Check-in to your hotel and freshen up.
Afternoon: Visit the Kodaikanal Lake and enjoy a leisurely walk around the lake. You can also rent a boat and enjoy the scenic views.
Evening: Take a stroll along Coaker's Walk, a scenic walking path that offers breathtaking views of the surrounding hills and valleys.
Recommended Food: Try some local South Indian cuisine, such as idlis, dosas, and vadas, at a nearby restaurant.
Estimated Daily Expense: ₹4,500 (accommodation: ₹2,000, food: ₹1,000, transportation: ₹1,000, activities: ₹500)
Travel Tips: Be sure to wear comfortable shoes and bring sunscreen and sunglasses to protect yourself from the sun.
Day 2: Sightseeing and Adventure

Morning: Visit Pillar Rocks, a unique rock formation that is a popular spot for trekking and photography.
Afternoon: Explore Bryant Park, a beautiful park that is home to a variety of flora and fauna. You can also visit the Kodaikanal Market to shop for some local handicrafts and souvenirs.
Evening: Enjoy a relaxing evening at your hotel or take a walk around the town to explore the local culture.
Recommended Food: Try some local Tamil Nadu specialties, such as parottas, chettinad chicken, and fish fry, at a nearby restaurant.
Estimated Daily Expense: ₹4,000 (accommodation: ₹2,000, food: ₹1,000, transportation: ₹500, activities: ₹500)
Travel Tips: Be sure to respect the local environment and wildlife, and avoid littering or damaging the natural beauty of the area.
Day 3: Waterfalls and Departure

Morning: Visit Silver Cascade, a stunning waterfall that is a popular spot for picnics and relaxation.
Afternoon: Return to your hotel, check-out, and depart for Madurai Airport or railway station (approximately 3-4 hours).
Evening: Depart from Madurai Airport or railway station.
Recommended Food: Try some local snacks, such as homemade chocolates and fresh fruits, at a nearby shop.
Estimated Daily Expense: ₹3,500 (accommodation: ₹0, food: ₹1,000, transportation: ₹1,500, activities: ₹1,000)
Travel Tips: Be sure to book your transportation in advance to avoid high costs, and try to avoid traveling during peak hours to avoid traffic congestion.
Total Estimated Budget: ₹12,000

This itinerary provides a mix of relaxation, sightseeing, and adventure, and should fit within your budget of ₹15,000. However, please note that the estimated costs are subject to change, and you should be prepared for any unexpected expenses. Additionally, be sure to respect the local environment and culture, and follow all necessary safety precautions to ensure a safe and enjoyable trip.

========================================

✅ Reviewer Feedback
Overall Rating: 8/10

Strengths:

The travel plan is well-structured and easy to follow.
The itinerary provides a good balance of relaxation, sightseeing, and adventure.
The budget breakdown is detailed and realistic.
The travel tips and recommendations are helpful and practical.
Weaknesses:

The itinerary is a bit rushed, with too many activities packed into each day.
The budget breakdown does not include any contingency planning for unexpected expenses.
The travel plan does not include any information about the family's specific interests or preferences.
The itinerary does not include any free time or flexibility to accommodate changes in plans.
Suggestions:

Consider adding an extra day to the itinerary to allow for more relaxation and flexibility.
Include some free time in the itinerary to allow the family to explore the town and its surroundings at their own pace.
Consider adding some activities or excursions that cater to the family's specific interests or preferences.
Include a contingency plan in the budget breakdown to account for any unexpected expenses.
Provide more information about the family's accommodation options, such as the type of hotel or resort, and the amenities included.
Consider including some information about the local culture and customs, such as dress codes or etiquette, to help the family prepare for their trip.
Additional Recommendations:

Consider booking a hotel or resort that offers a package deal, including meals and activities, to help save on costs.
Look into booking a guided tour or excursion to help the family make the most of their time in Kodaikanal.
Consider purchasing travel insurance to protect against any unexpected medical or travel-related expenses.
Make sure to research and book any necessary transportation or activities in advance to avoid high costs or availability issues.
Missing Recommendations:

Information about the family's specific interests or preferences.
Contingency planning for unexpected expenses.
Free time or flexibility in the itinerary.
Information about the local culture and customs.
Recommendations for shopping or dining options.
Overall, the travel plan is well-structured and provides a good balance of relaxation, sightseeing, and adventure. However, it could benefit from some additional planning and flexibility to accommodate the family's specific needs and preferences.

========================================

Right now every agent calls the LLM separately.

Planner
   ↓
Research
   ↓
Budget
   ↓
Itinerary
   ↓
Reviewer

This works, but the agents are not specialized. The LLM is estimating everything from its general knowledge.

Version 1 ✅ (Completed)
5 Agents
Simple Gradio Interface
Single Response